In [3]:
import os
import time
import shutil

os.system("docker rm -f qdrant-container > /dev/null 2>&1")

print("\n🐳 Đang khởi động container Qdrant mới...")
os.system("""
docker run -d \
    --name qdrant-container \
    -p 6333:6333 \
    -p 6334:6334 \
    -v "$(pwd)/qdrant_storage:/qdrant/storage:z" \
    qdrant/qdrant:latest
""")
print("✅ Qdrant đã sẵn sàng. Chờ 2 giây để khởi động hoàn tất...")
time.sleep(2)


🐳 Đang khởi động container Qdrant mới...
71426e0cdb3a131fec8e75654d958b605027e87d8ed03b9289e04e606b964235
✅ Qdrant đã sẵn sàng. Chờ 2 giây để khởi động hoàn tất...


In [4]:
# IMPORT PACKAGES
import os
import json
from pathlib import Path
from typing import List, Tuple
import time
import uuid
import numpy as np
import pandas as pd
from langchain.schema import Document
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_google_genai import ChatGoogleGenerativeAI
from sentence_transformers import CrossEncoder
from qdrant_client import QdrantClient
from qdrant_client.http.models import Distance, VectorParams, PointStruct
from langchain_openai import ChatOpenAI 
from config import (
    DEEPSEEK_API_KEY,
    QDRANT_HOST,
    QDRANT_PORT,
    COLLECTION_NAME,
    DELAY_BETWEEN_CALLS,
    EMBEDDING_MODEL_NAME,
    EMBEDDING_DEVICE,
    LIST_GOOGLE_API_KEYS,
    RERANKER_MODEL_NAME,
    RERANKER_DEVICE
)
import warnings
warnings.filterwarnings("ignore")
warnings.filterwarnings(
    "ignore", 
    message="flash_attn is not installed. Using PyTorch native attention implementation."
)

# Generate Evaluation Dataset (Full Flow Advanced RAG + Get Context)

- ChatGPT OSS: Response 
- Gemini 2.5 Pro: Parsing

In [ ]:
# --- KHỞI TẠO CÁC MODEL ---
print("🧠 Đang khởi tạo các model")
from langchain_together import ChatTogether

# Embedding model
embedding_model = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL_NAME,
    model_kwargs={"device": EMBEDDING_DEVICE, "trust_remote_code": True}
)

# LLM
try:
    deepseek_llm_client = ChatOpenAI(
        model="deepseek-chat",
        api_key=DEEPSEEK_API_KEY,
        base_url="https://api.deepseek.com/v1",
        temperature=0.3,    
    )
    print("✅ DeepSeek client đã sẵn sàng.")
except Exception as e:
    print(f"❌ Lỗi khi khởi tạo DeepSeek client: {e}")
    deepseek_llm_client = None
# ChatGPT OSS
try: 
    chatgpt_oss_client = ChatTogether(
        together_api_key="",
        model="openai/gpt-oss-120b",
    )
    print("✅ ChatGPT OSS client đã sẵn sàng.")
except Exception as e:
    print(f"❌ Lỗi khi khởi tạo ChatGPT OSS client: {e}")
    chatgpt_oss_client = None
# Reranker model
reranker_model = CrossEncoder(model_name_or_path= RERANKER_MODEL_NAME,
                              device=RERANKER_DEVICE,
                              trust_remote_code=True)
print("✅ Reranker model đã sẵn sàng.")
qdrant_client = QdrantClient(host=QDRANT_HOST, port=QDRANT_PORT)
print("✅ Các model đã sẵn sàng.")

# Clear previous collection of Qdrant if exists
def clear_qdrant_collection():
    """Xóa collection để đảm bảo dữ liệu mới mỗi lần chạy."""
    try:
        qdrant_client.delete_collection(collection_name=COLLECTION_NAME)
        print(f"✅ Đã xóa collection '{COLLECTION_NAME}' (nếu tồn tại).")
    except Exception:
        pass 
# Setup Qdrant collection 
def setup_qdrant_collection(embedding_dim: int):
    """Tạo hoặc tái tạo collection trong Qdrant."""
    try:
        qdrant_client.recreate_collection(
            collection_name=COLLECTION_NAME,
            vectors_config=VectorParams(size=embedding_dim, distance=Distance.COSINE)
        )
        print(f"✅ Đã tạo/tái tạo collection '{COLLECTION_NAME}' trong Qdrant.")
        return True
    except Exception as e:
        print(f"❌ Lỗi khi setup Qdrant collection: {e}")
        return False

# Store chunks in Qdrant
def store_child_chunks_in_qdrant(child_chunks: List[Document], child_embeddings: np.ndarray):
    """Lưu child chunks và embeddings vào Qdrant."""
    points = []
    for chunk, emb in zip(child_chunks, child_embeddings):
        payload = {
            "content": chunk.page_content,
            "parent_id": chunk.metadata.get("parent_id"),
            "parent_content": chunk.metadata.get("parent_content"),
            "source": chunk.metadata.get("source", "unknown")
        }
        points.append(PointStruct(id=str(uuid.uuid4()), vector=emb.tolist(), payload=payload))
    qdrant_client.upsert(collection_name=COLLECTION_NAME, points=points, wait=True)
    print(f"✅ Đã lưu {len(points)} child chunks vào Qdrant.")

# Parent - Child chunking strategy and Embedding in Qdrant
def setup_small_to_big_data_with_qdrant(markdown_content: str):
    """Chuẩn bị dữ liệu cho chiến lược Small-to-Big và lưu vào Qdrant."""
    print("--- Bước 1: Chuẩn bị dữ liệu Small-to-Big với Qdrant ---")
    parent_splitter = RecursiveCharacterTextSplitter(chunk_size=2000, chunk_overlap=200)
    child_splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=50)
    parent_chunks = parent_splitter.create_documents([markdown_content])
    print(f"✅ Đã tạo {len(parent_chunks)} parent chunks (đoạn văn lớn).")

    all_child_chunks = []
    for i, p_chunk in enumerate(parent_chunks):
        child_texts = child_splitter.split_text(p_chunk.page_content)
        for text in child_texts:
            all_child_chunks.append(Document(page_content=text, metadata={"parent_id": i, "parent_content": p_chunk.page_content}))
    print(f"✅ Đã tạo {len(all_child_chunks)} child chunks (câu nhỏ) từ các parent.")
            
    sample_embedding = embedding_model.embed_query("test")
    if not setup_qdrant_collection(len(sample_embedding)): 
        return None, False
    
    print("🧠 Đang embedding các child chunks...")
    child_contents = [c.page_content for c in all_child_chunks]
    child_embeddings = np.array(embedding_model.embed_documents(child_contents))
    print("✅ Embedding child chunks hoàn tất!")
    
    store_child_chunks_in_qdrant(all_child_chunks, child_embeddings)
    return parent_chunks, True

def retrieve_context_list(query: str, top_k_final: int = 3, retrieve_k_children: int = 15) -> List[str]:
    """
    Pipeline RAG: Search child chunks --> Mapping parents --> Reranking parents --> Return top-k parents.
    """
    query_embedding = embedding_model.embed_query(query)
    search_results = qdrant_client.search(collection_name=COLLECTION_NAME, query_vector=query_embedding, limit=retrieve_k_children, with_payload=True)
    
    candidate_parents = {hit.payload["parent_content"] for hit in search_results if hit.payload and "parent_content" in hit.payload}
    
    if not candidate_parents: return [] 
    
    parent_contents = list(candidate_parents)
    pairs = [(query, content) for content in parent_contents]
    scores = reranker_model.predict(pairs)
    reranked_results = sorted(zip(scores, parent_contents), key=lambda x: x[0], reverse=True)
    
    final_docs = [content for _, content in reranked_results[:top_k_final]]
    return final_docs
    
def answer_with_context(query: str, context_list: List[str]) -> str: 
    """Sử dụng LLM để trả lời câu hỏi dựa trên context được cung cấp."""
    
    full_context_str = "\n\n---\n\n".join(context_list)
    
    prompt_template = f"""Bạn là một trợ lý AI chuyên nghiệp, chỉ trả lời dựa trên context được cung cấp.
    Trả lời câu hỏi một cách ngắn gọn và chính xác. Nếu không có thông tin trong context, hãy nói "Tôi không tìm thấy thông tin trong tài liệu."

    Context:
    ---
    {full_context_str}
    ---
    

    Câu hỏi: {query}

    Trả lời:
    """
    response = chatgpt_oss_client.invoke(prompt_template)
    return response.content

# --- CHẠY PIPELINE VÀ THU THẬP DỮ LIỆU ---

def main():
    """
    Hàm chính để thực thi toàn bộ pipeline RAG và đánh giá.
    """
    generated_data_for_eval = [] 
    markdown_file_path = Path("backend/output_parsing/gemini-2.5-raw-input.md")

    if not markdown_file_path.exists():
        print(f"❌ Lỗi: Không tìm thấy file {markdown_file_path}. Vui lòng chạy bước parsing trước.")
        return 

    with open(markdown_file_path, 'r', encoding='utf-8') as f:
        markdown_content = f.read()

    clear_qdrant_collection()
    parent_chunks, success = setup_small_to_big_data_with_qdrant(markdown_content)
    
    if not success:
        print("❌ Không thể setup dữ liệu với Qdrant. Dừng chương trình.")
        return 

    # Lấy câu hỏi từ file ragasdf_base.json
    with open(Path("backend/schemas/ragasdf_base.json"), 'r', encoding='utf-8') as f:
        queries_data = json.load(f)
    queries = [item['question'] for item in queries_data]

    print(f"\n\n=== 🚀 BẮT ĐẦU CHẠY RAG VÀ THU THẬP DỮ LIỆU TRÊN {len(queries)} CÂU HỎI ===")

    for i, query in enumerate(queries, 1):
        # ... (phần code trong vòng lặp for của bạn giữ nguyên)
        print(f"\n--- Query {i}/{len(queries)} ---")
        print(f"❓ Câu hỏi: {query}")
        
        context_list = retrieve_context_list(query)
        answer = answer_with_context(query, context_list) 
        print(f"✅ Câu trả lời của hệ thống: {answer}")
        
        generated_data_for_eval.append({
            "question": query,
            "answer": answer,
            "contexts": context_list 
        })
        
        print("-" * 40)
        if i < len(queries):
            print(f"⏳ Đợi {DELAY_BETWEEN_CALLS} giây...")
            time.sleep(DELAY_BETWEEN_CALLS)
    
    print("\n\n✅ Đã thu thập xong toàn bộ dữ liệu (answer, context).")
    display(pd.DataFrame(generated_data_for_eval).head())
    
    output_file = Path('output_parsing/generated_data_for_eval(full_raw_input_gemini2.5_parsing_chatoss).json')
    output_file.parent.mkdir(parents=True, exist_ok=True)
    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(generated_data_for_eval, f, ensure_ascii=False, indent=4)
    print(f"✅ Đã lưu kết quả vào file: {output_file}")


if __name__ == "__main__":
    main()

🧠 Đang khởi tạo các model


flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn i

✅ DeepSeek client đã sẵn sàng.
✅ ChatGPT OSS client đã sẵn sàng.
✅ Reranker model đã sẵn sàng.
✅ Các model đã sẵn sàng.
✅ Đã xóa collection 'child_chunks_eval_notebook' (nếu tồn tại).
--- Bước 1: Chuẩn bị dữ liệu Small-to-Big với Qdrant ---
✅ Đã tạo 17 parent chunks (đoạn văn lớn).
✅ Đã tạo 97 child chunks (câu nhỏ) từ các parent.
✅ Đã tạo/tái tạo collection 'child_chunks_eval_notebook' trong Qdrant.
🧠 Đang embedding các child chunks...


KeyboardInterrupt: 

# RAGAS Evaluation

In [ ]:
from ragas_evaluation import RagasEvaluator
import pandas as pd
from pathlib import Path
import json

# 1. Khởi tạo Evaluator
evaluator = RagasEvaluator(list_of_api_keys=LIST_GOOGLE_API_KEYS)

# Load generated data từ file JSON: (generated data is RAG output: question, LLM answer, context)
generated_data_file = Path('output_parsing/generated_data_for_eval(full_raw_input_claude_parsing_deepseek).json')
if not generated_data_file.exists():
    print(f"❌ Lỗi: Không tìm thấy file {generated_data_file}. Vui lòng chạy bước thu thập dữ liệu trước.")
else:
    with open(generated_data_file, 'r', encoding='utf-8') as f:
        generated_data_for_eval = json.load(f)
        
# 2.  Load base data (chứa question và ground_truth)
base_data = evaluator.load_base_dataset(Path('backend/schemas/ragasdf_base.json'))

# 3. Chuẩn bị dataset hoàn chỉnh bằng cách kết hợp base_data và generated_data
'''
Evaluation dataset includes:
- question
- ground_truth (optional but need for some metrics: answer_correctness)
- contexts (list of context strings)
- answer (LLM generated answer)
'''
eval_dataset = evaluator.prepare_evaluation_dataset(
    base_data=base_data,
    generated_data=generated_data_for_eval 
)
    

# 4. Xuất file dataset RAGAS hoàn chỉnh ra Excel để kiểm tra
if eval_dataset:
    ragas_df = eval_dataset.to_pandas()
    ragas_df['context_count'] = ragas_df['contexts'].apply(len)
    output_df_path = "ragas_dataset_for_evaluation(full_raw_input_claude_parsing_deepseek).xlsx"
    ragas_df.to_excel(output_df_path, index=False)
    print(f"💾 Đã xuất file dataset RAGAS hoàn chỉnh ra: {output_df_path}")
    
# 5. Chạy đánh giá
if eval_dataset:
    results_df = evaluator.run_evaluation(eval_dataset)

    if results_df is not None:
        report_dir = "evaluation_reports"
        report_filename = "ragas_report(full_raw_input_claude_parsing_deepseek).xlsx"
        saved_path = evaluator.save_report(results_df, output_dir=report_dir, filename=report_filename)
        if saved_path:
            print(f"✅ Báo cáo đã được lưu tại: {saved_path}")
        else:
            print("❌ Không thể lưu báo cáo.")
        evaluator.display_evaluation_cost()
    else:
        print("❌ Đánh giá không trả về kết quả.")
else:
    print("❌ Không thể tạo dataset để đánh giá.")


# RAGAS Evaluation (ChatGPT OSS 120B)

In [1]:
# Querying chat models with Together AI

from langchain_together import ChatTogether

# choose from our 50+ models here: https://docs.together.ai/docs/inference-models
chat = ChatTogether(
    together_api_key="ee493f081e8bad724f15cbcc32d72ed090beadc0af0a0532e44715a5e5a1cee0",
    model="openai/gpt-oss-120b",
)



# if you don't want to do streaming, you can use the invoke method
chat.invoke("Tell me fun things to do in NYC")

AIMessage(content='### 🎉 20 Fun Things to Do in New\u202fYork\u202fCity (for any mood, budget, or season)\n\n| # | Activity | Why It’s Fun | Neighborhood / Area | Approx. Time | Cost* |\n|---|----------|--------------|----------------------|--------------|-------|\n| 1 | **Statue of Liberty & Ellis Island** | Iconic photo‑ops, history, great harbor views | Battery Park (ferry) | Half‑day | $24–$30 (ferry & pedestal) |\n| 2 | **Walk the High Line** | Elevated park with art installations and river vistas | West Side (Chelsea → Hudson Yards) | 1–2\u202fhrs | Free |\n| 3 | **Visit the Met (Metropolitan Museum of Art)** | World‑class collections, rooftop garden with skyline view | Upper East Side | 2–4\u202fhrs | $30 (pay‑what‑you‑wish for NY residents) |\n| 4 | **Catch a Broadway Show** | Live theater magic; many discounted same‑day tickets at TKTS | Times Square | 2.5–3\u202fhrs | $50–$250 (TKTS $30–$75) |\n| 5 | **Explore Central\u202fPark** | Row a boat, visit the zoo, or just people‑wa

In [5]:
pip install langchain_together

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Note: you may need to restart the kernel to use updated packages.
